In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Feature Engineering

In [2]:
path = "../../data/time_series_60.csv"
df = pd.read_csv(path)

filtered_cols = ['utc_timestamp', 'DE_load_actual_entsoe_transparency']
df = df[filtered_cols]

df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"], utc=True)
df = df.set_index("utc_timestamp")

df.head()

,DE_load_actual_entsoe_transparency
utc_timestamp,
2014-12-31 23:00:00+00:00,NaN
2015-01-01 00:00:00+00:00,41151.0
2015-01-01 01:00:00+00:00,40135.0
2015-01-01 02:00:00+00:00,39106.0
2015-01-01 03:00:00+00:00,38765.0


# Cyclical encoding

In [3]:
# Hour of day (strongest effect)
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

# Day of week (work vs weekend)
df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)

df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

df['month_sin'] = np.sin(2 * np.pi * (df.index.month / 12))
df['month_cos'] = np.cos(2 * np.pi * (df.index.month / 12))

days_in_year = df.index.is_leap_year.astype(int) + 365
df['dayofyear_sin'] = np.sin(2 * np.pi * df.index.dayofyear / days_in_year)
df['dayofyear_cos'] = np.cos(2 * np.pi * df.index.dayofyear / days_in_year)

week = df.index.isocalendar().week.astype(int)
df['weekofyear_sin'] = np.sin(2 * np.pi * week / 52)
df['weekofyear_cos'] = np.cos(2 * np.pi * week / 52)

# Lag features

In [4]:
target = "DE_load_actual_entsoe_transparency"

# Lag features up until 1 week into past
for lag in [1, 2, 3, 6, 12, 24, 48, 72, 168]:
    df[f'{target}_lag_{lag}'] = df[target].shift(lag)

# Rolling features 1 day and 1 week into past
for w in [24, 168]:
    min_p = int(w * 0.5) 
    df[f'{target}_roll_mean_{w}'] = df[target].shift(1).rolling(w, min_periods=min_p).mean() # Calculate rolling features
    df[f'{target}_roll_std_{w}']  = df[target].shift(1).rolling(w, min_periods=min_p).std()

# Ramp features
df['ramp_1h'] = df[target].diff(1)
df['ramp_24h'] = df[target].diff(24)

In [5]:
df.tail()

,DE_load_actual_entsoe_transparency,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,month_sin,month_cos,dayofyear_sin,dayofyear_cos,...,DE_load_actual_entsoe_transparency_lag_24,DE_load_actual_entsoe_transparency_lag_48,DE_load_actual_entsoe_transparency_lag_72,DE_load_actual_entsoe_transparency_lag_168,DE_load_actual_entsoe_transparency_roll_mean_24,DE_load_actual_entsoe_transparency_roll_std_24,DE_load_actual_entsoe_transparency_roll_mean_168,DE_load_actual_entsoe_transparency_roll_std_168,ramp_1h,ramp_24h
utc_timestamp,,,,,,,,,,,,,,,,,,,,,
2020-09-30 19:00:00+00:00,57559.0,-0.965926,0.258819,0.974928,-0.222521,0,-1.0,-1.836970e-16,-0.999963,-0.008583,...,56775.0,56994.0,47286.0,55434.0,57659.708333,8059.157743,54104.535714,9028.791506,-3618.0,784.0
2020-09-30 20:00:00+00:00,54108.0,-0.866025,0.500000,0.974928,-0.222521,0,-1.0,-1.836970e-16,-0.999963,-0.008583,...,53356.0,53622.0,46219.0,51400.0,57692.375000,8057.004416,54117.184524,9032.153035,-3451.0,752.0
2020-09-30 21:00:00+00:00,49845.0,-0.707107,0.707107,0.974928,-0.222521,0,-1.0,-1.836970e-16,-0.999963,-0.008583,...,49467.0,49617.0,43398.0,47243.0,57723.708333,8040.853288,54133.303571,9029.690883,-4263.0,378.0
2020-09-30 22:00:00+00:00,46886.0,-0.500000,0.866025,0.974928,-0.222521,0,-1.0,-1.836970e-16,-0.999963,-0.008583,...,46652.0,46430.0,41407.0,44914.0,57739.458333,8024.330545,54148.791667,9020.027941,-2959.0,234.0
2020-09-30 23:00:00+00:00,45461.0,-0.258819,0.965926,0.974928,-0.222521,0,-1.0,-1.836970e-16,-0.999963,-0.008583,...,44895.0,44632.0,40414.0,44332.0,57749.208333,8010.403019,54160.529762,9009.215041,-1425.0,566.0
